In [ ]:
import time

import torch
from torch import nn
from torch.nn import functional as F

import numpy as np

import tqdm as tqdm

import tiktoken

from model import ModelConfig, MyGPT
from dataloader import DataLoader

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

# Define tokens  and their encode/decode

In [ ]:
# TokenizerName = 'Char'
TokenizerName = 'Tiktoken'

if TokenizerName=='Char':

    # TODO

    # set(text): find unique characteres
    # list(set(text)): convert to list
    # sorted(list(set(text))): sort
    Tokens = sorted(list(set(text)))
    NToken = len(Tokens)

    print('Number of tokens:', NToken)

    # encode: encode(list of character) = list of integer, index from Tokens list
    # decode: decode(list of integer) = list of chracter

    dict_c_to_i = {c:i_c for i_c, c in enumerate(Tokens)}
    dict_i_to_c = {i_c:c for i_c, c in enumerate(Tokens)}

    encode = lambda list_c: [dict_c_to_i[c] for c in list_c]
    decode = lambda list_i: ''.join([dict_i_to_c[i] for i in list_i])
    

if TokenizerName=='Tiktoken':
    
    enc = tiktoken.get_encoding("gpt2")

    NToken = enc.n_vocab

    encode = lambda list_c: enc.encode(list_c)
    decode = lambda list_i: enc.decode(list_i)


In [ ]:
print('Number of Tokens:',NToken)
print(encode('My name is Jaesung'))
print(decode(encode('My name is Jaesung')))

# Now declare our model

In [ ]:
_config = ModelConfig()

_config.DToken = NToken

_config.ContextSize = 1024

_config.DHead = 64

_config.NHeadQ = 12
_config.NHeadKV = 12

_config.NLayer = 12


In [ ]:
my_model = MyGPT(_config).to(device)

my_model = torch.compile(my_model)

In [ ]:
# print the number of parameters in the model
print(sum(p.numel() for p in my_model.parameters()), 'parameters')

# Generate with the initial model

In [ ]:
input_text = 'Dog'

tmp_start_text = encode(input_text)

# generate from the model
print('Input text:')
print(input_text)

output_text = decode(my_model.generate(tmp_start_text, max_tokens=10)[0].tolist())
print('Output text:')
print(output_text)

# Prepare training

In [ ]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(my_model.parameters(), lr=3E-4)

In [ ]:
# Size of the batch for each training
NBatch = 4

# Max iteration of the training
NMaxIteration = 1000

# Logging
LogEvery = 50

# Evaluate the training every
EvalEvery = 100
# Number of iteration for every evaluation step
EvalIteration = 1

# Learning rate
LearningRate = 3E-4

# Prepare data

In [ ]:
InputDataType = 'shakespeare'

data_dir = f'data/{InputDataType}'

map_dls = dict()
for split in ['train', 'val']:
    map_dls[split] = DataLoader(NBatch, _config.ContextSize, data_dir, split)


In [ ]:
# Function decorator
@torch.no_grad()
def estimate_loss():
    out = {}
    my_model.eval()
    for split in ['train', 'val']:
        # Make tensor placeholder
        losses = torch.zeros(EvalIteration)
        for k in range(EvalIteration):
            X, Y = map_dls[split].next_batch()
            X, Y = X.to(device), Y.to(device)
            loss = my_model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    my_model.train()
    return out

In [ ]:
for step in range(NMaxIteration):

    if step % EvalEvery == 0 or step == NMaxIteration - 1:
        losses = estimate_loss()
        print(f"step {step}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    t0 = time.time()
    last_step = (step == NMaxIteration - 1)

    # sample a batch of data
    xb, yb = map_dls['train'].next_batch()
    xb, yb = xb.to(device), yb.to(device)

    # evaluate the loss
    loss = my_model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    t1 = time.time()
    dt = t1 - t0 # time difference in seconds

    tokens_processed = NBatch * _config.ContextSize
    tokens_per_sec = tokens_processed / dt

    if step % LogEvery == 0 or step == NMaxIteration - 1:
        print(f"step {step:5d} | loss: {loss.item():.6f} dt: {dt*1000:.2f}ms | tok/sec: {tokens_per_sec:.2f}")

    if device=='mps':
        torch.mps.synchronize()

In [ ]:
map_dls['train'].NFullLoop

In [ ]:
input_text = 'Hello'

tmp_start_text = encode(input_text)

# generate from the model
print('Input text:')
print(input_text)

output_text = decode(my_model.generate(tmp_start_text, max_tokens=10)[0].tolist())
print('Output text:')
print(output_text)
